# trajectory-kit: Advanced Tutorial

This tutorial assumes you've worked through `tutorial.ipynb`. Same synthetic
data, same three methods (`positions()`, `fetch()`, `select()`) — but now
with `devFlag=True`, which exposes the full structured result envelope.

You'll learn to:

- Read the five-key envelope returned with `devFlag=True`
- Understand the **selection** block — which atoms came from where
- Read **metadata** about loaded files
- Use the **planner** to estimate payload size before reading
- Use the **combined** estimate for multi-domain `fetch()` calls
- Use `planFlag=True` to skip the read entirely and just get the plan

Everything below is opt-in — you only see this complexity if you ask for it.


## 0. Setup

In [1]:
import struct
import tempfile
from pathlib import Path
import numpy as np

# Synthetic files are written to a fresh temp directory. The path is printed
# below so you can browse to it and inspect the raw files if you want to.
tmp = Path(tempfile.mkdtemp(prefix="trajkit_tutorial_"))

# ── 20-atom synthetic system ──────────────────────────────────────────────
# Protein (PROT): ALA(6) + GLY(5) + PRO(3) = 14 atoms, 3 residues
# Solvent (SOLV): 2 x TIP3 water molecules = 6 atoms, 2 residues
#
# Bond graph (17 bonds):
#   Backbone: N-CA-C chain with peptide bonds
#   ALA sidechain: CA-CB
#   Carbonyl: C=O on each residue
#   PRO N closes the chain
#   Waters: each OH2 bonded to H1 and H2
#
# Degree distribution: 10 x degree-1, 6 x degree-2, 4 x degree-3
# Atom types used: NH1, H, CT1, CT3, C, O, CT2, N, CP1, OT, HT

pdb_text = """\
ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT
ATOM      4  CB  ALA A   1       3.000   6.300   5.200  1.00  0.00      PROT
ATOM      5  C   ALA A   1       3.200   3.900   4.600  1.00  0.00      PROT
ATOM      6  O   ALA A   1       2.800   2.800   4.400  1.00  0.00      PROT
ATOM      7  N   GLY A   2       4.600   4.100   4.500  1.00  0.00      PROT
ATOM      8  HN  GLY A   2       5.000   5.000   4.700  1.00  0.00      PROT
ATOM      9  CA  GLY A   2       5.800   3.200   4.100  1.00  0.00      PROT
ATOM     10  C   GLY A   2       7.200   3.500   4.000  1.00  0.00      PROT
ATOM     11  O   GLY A   2       7.700   4.600   4.200  1.00  0.00      PROT
ATOM     12  N   PRO A   3       8.100   2.500   3.700  1.00  0.00      PROT
ATOM     13  CA  PRO A   3       9.500   2.800   3.600  1.00  0.00      PROT
ATOM     14  C   PRO A   3      10.300   1.900   3.200  1.00  0.00      PROT
ATOM     15  OH2 TIP B   4      12.000   5.000   5.000  1.00  0.00      SOLV
ATOM     16  H1  TIP B   4      12.900   5.500   5.100  1.00  0.00      SOLV
ATOM     17  H2  TIP B   4      11.300   5.700   4.800  1.00  0.00      SOLV
ATOM     18  OH2 TIP B   5      14.000   3.000   5.500  1.00  0.00      SOLV
ATOM     19  H1  TIP B   5      14.800   3.600   5.400  1.00  0.00      SOLV
ATOM     20  H2  TIP B   5      13.400   3.700   5.200  1.00  0.00      SOLV
END
"""

psf_text = """\
PSF

       1 !NTITLE
 REMARKS synthetic 20-atom ALA-GLY-PRO + 2xTIP3

      20 !NATOM
       1 PROT     1 ALA  N    NH1     -0.47000     14.007           0
       2 PROT     1 ALA  HN   H        0.31000      1.008           0
       3 PROT     1 ALA  CA   CT1     -0.02000     12.011           0
       4 PROT     1 ALA  CB   CT3     -0.27000     12.011           0
       5 PROT     1 ALA  C    C        0.51000     12.011           0
       6 PROT     1 ALA  O    O       -0.51000     15.999           0
       7 PROT     2 GLY  N    NH1     -0.47000     14.007           0
       8 PROT     2 GLY  HN   H        0.31000      1.008           0
       9 PROT     2 GLY  CA   CT2     -0.18000     12.011           0
      10 PROT     2 GLY  C    C        0.51000     12.011           0
      11 PROT     2 GLY  O    O       -0.51000     15.999           0
      12 PROT     3 PRO  N    N       -0.29000     14.007           0
      13 PROT     3 PRO  CA   CP1     -0.02000     12.011           0
      14 PROT     3 PRO  C    C        0.51000     12.011           0
      15 SOLV     4 TIP  OH2  OT      -0.83400     15.999           0
      16 SOLV     4 TIP  H1   HT       0.41700      1.008           0
      17 SOLV     4 TIP  H2   HT       0.41700      1.008           0
      18 SOLV     5 TIP  OH2  OT      -0.83400     15.999           0
      19 SOLV     5 TIP  H1   HT       0.41700      1.008           0
      20 SOLV     5 TIP  H2   HT       0.41700      1.008           0

      17 !NBOND: bonds
       1       2       1       3       3       4       3       5
       5       6       5       7       7       8       7       9
       9      10      10      11      10      12      12      13
      13      14      15      16      15      17      18      19
      18      20

       0 !NTHETA: angles

       0 !NPHI: dihedrals

       0 !NIMPHI: impropers

"""

def write_dcd(path, n_atoms=20, n_frames=5):
    """5-frame DCD. Each frame shifts all atoms by +1 A along x."""
    def record(payload):
        n = struct.pack('<i', len(payload))
        return n + payload + n
    icntrl = [0] * 20
    icntrl[0] = n_frames
    hdr = b'CORD' + struct.pack('<20i', *icntrl)
    hdr += b'\x00' * (84 - len(hdr))
    title = struct.pack('<i', 1) + b'synthetic 20-atom trajectory    '
    natom = struct.pack('<i', n_atoms)
    x0 = np.array([ 1.0, 0.1, 2.4, 3.0, 3.2, 2.8,
                    4.6, 5.0, 5.8, 7.2, 7.7, 8.1,
                    9.5,10.3,12.0,12.9,11.3,14.0,14.8,13.4], dtype=np.float32)
    y0 = np.array([ 5.0, 5.8, 5.0, 6.3, 3.9, 2.8,
                    4.1, 5.0, 3.2, 3.5, 4.6, 2.5,
                    2.8, 1.9, 5.0, 5.5, 5.7, 3.0, 3.6, 3.7], dtype=np.float32)
    z0 = np.array([ 5.0, 5.3, 5.0, 5.2, 4.6, 4.4,
                    4.5, 4.7, 4.1, 4.0, 4.2, 3.7,
                    3.6, 3.2, 5.0, 5.1, 4.8, 5.5, 5.4, 5.2], dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(record(hdr))
        f.write(record(title))
        f.write(record(natom))
        for fi in range(n_frames):
            xs = x0 + float(fi)
            f.write(record(xs.tobytes()))
            f.write(record(y0.tobytes()))
            f.write(record(z0.tobytes()))

pdb_path = tmp / 'synthetic.pdb'
psf_path = tmp / 'synthetic.psf'
dcd_path = tmp / 'synthetic.dcd'

pdb_path.write_text(pdb_text, encoding='ascii')
psf_path.write_text(psf_text, encoding='ascii')
write_dcd(dcd_path)

print('Synthetic files written to:', tmp)
print('  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)')
print('  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)')
print('  DCD: 5 frames - x coordinates shift +1 A per frame')


Synthetic files written to: /var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_l6zuvb0p
  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)
  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)
  DCD: 5 frames - x coordinates shift +1 A per frame


In [2]:
from trajectory_kit import sim

s = sim(
    typing=pdb_path,
    topology=psf_path,
    trajectory=dcd_path,
)


## 1. The `devFlag` — see what's happening

In the basic tutorial, `positions()` returned a plain ndarray and `fetch()`
returned a plain dict. Pass `devFlag=True` and you get a structured
**envelope** instead — a 5-key dict with everything `sim` knows about the
call.


In [3]:
out = s.positions(TYPE_Q={'atom_name': 'CA'}, devFlag=True)

print('keys:', list(out.keys()))
print('mode:', out['mode'])
print('payload shape:', out['payload'].shape)


keys: ['mode', 'selection', 'metadata', 'plan', 'payload']
mode: positions
payload shape: (5, 3, 3)


The five keys, in summary:

| Key | What it tells you |
|-----|-------------------|
| `mode` | Which method you called: `"positions"`, `"fetch"`, or `"select"`. |
| `selection` | Per-domain breakdown of which atoms each query contributed. |
| `metadata` | Per-loaded-file facts: atom counts, frame counts, file size, etc. |
| `plan` | Per-domain cost estimate, plus a cross-domain `combined` rollup. |
| `payload` | The actual data — same shape as without `devFlag`. |

The next sections walk through each one.


## 2. The `selection` block

`selection` records, for each domain, whether you supplied a query, whether
that query produced an id list, and how many atoms it matched. It also
records the cross-domain merge mode and the final intersected count.


In [4]:
import pprint

out = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TOPO_Q={'charge': (-1.0, 1.0)},
    devFlag=True,
)

pprint.pp(out['selection'])


{'merge_mode': 'intersection',
 'typing': {'query_provided': True, 'ids_provided': True, 'n_matched': 3},
 'topology': {'query_provided': True, 'ids_provided': True, 'n_matched': 20},
 'trajectory': {'query_provided': False,
                'ids_provided': False,
                'n_matched': None},
 'resolved_count': 3}


Field meanings:

| Field | Meaning |
|-------|---------|
| `merge_mode` | `"intersection"` for `positions()` / `fetch()`; `"none"` for `select()`. |
| `<domain>.query_provided` | Did you pass a query dict for this domain? |
| `<domain>.ids_provided` | Did this domain contribute a `global_ids` list to the intersection? |
| `<domain>.n_matched` | Size of the id list this domain contributed (or `None` if it didn't). |
| `resolved_count` | Final size of the intersected selection — the atom count in your payload. |

`ids_provided` can be `False` even when `query_provided` is `True`. Trajectory
formats like DCD have no per-atom data to filter on, so they participate in
the call but don't add an id constraint.


## 3. The `metadata` block

`metadata` is **query-independent** — it reflects every file currently
loaded, not what you asked about. It lets you check file-level facts without
issuing a separate call.


In [5]:
out = s.positions(TYPE_Q={'atom_name': 'CA'}, devFlag=True)
pprint.pp(out['metadata'])


{'typing': {'source': 'typing',
            'file_path': '/var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_l6zuvb0p/synthetic.pdb',
            'file_format': 'pdb',
            'file_size_bytes': 1544,
            'n_atoms': 20,
            'n_residues': 5,
            'box_size': (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)},
 'topology': {'source': 'topology',
              'file_path': '/var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_l6zuvb0p/synthetic.psf',
              'file_format': 'psf',
              'file_size_bytes': 1871,
              'n_atoms': 20,
              'n_residues': 5},
 'trajectory': {'source': 'trajectory',
                'file_path': '/var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_l6zuvb0p/synthetic.dcd',
                'file_format': 'dcd',
                'file_size_bytes': 1468,
                'n_atoms': 20,
                'n_frames': 5}}


Each domain's metadata block has three tiers:

- **Tier 1** (always present): `source`, `file_path`, `file_format`,
  `file_size_bytes`, `n_atoms`, and `n_frames` for trajectory files.
- **Tier 2** (present when meaningful): `n_residues`, `n_segments`,
  `box_size`, `timestep`, etc. — surfaced only when the parser reported
  them.
- **Tier 3** (`format_specific`): a sub-dict of anything else the parser
  reported that doesn't fit elsewhere. Empty for most calls.


## 4. The `plan` block — predicting cost before reading

`plan` is **query-dependent** — it only includes domains that contributed
to the call. Each per-domain entry estimates how big the payload will be.

There are two planner modes:

- **`"header"`** — the planner reads a fixed-size file header (e.g. DCD's
  binary header) and returns exact counts. Cheap and accurate.
- **`"stochastic"`** — the planner samples records (e.g. PDB ATOM lines)
  and extrapolates. Approximate; carries a `confidence` field.

Both produce the same key fields: `n_atoms`, `n_frames`,
`bytes_per_atom_per_frame`, and a derived `estimated_bytes`.


In [6]:
out = s.positions(TYPE_Q={'atom_name': 'CA'}, devFlag=True)
pprint.pp(out['plan'])


{'trajectory': {'planner_mode': 'header',
                'source': 'trajectory',
                'file_format': 'dcd',
                'request': 'positions',
                'n_atoms': 20,
                'n_frames': 5,
                'bytes_per_atom_per_frame': 12,
                'estimated_bytes': 1200},
 'combined': {'n_atoms_upper_bound': 20, 'total_estimated_bytes': 1200}}


Notice `plan` has both per-domain entries (`'trajectory'` here) and a
`'combined'` rollup. Per-domain plans tell you what *each* parser estimates
in isolation; combined tells you what the cross-domain intersection actually
costs.

The fields per per-domain plan:

| Field | Meaning |
|-------|---------|
| `planner_mode` | `"header"` or `"stochastic"`. |
| `source` | Which domain (`"typing"` / `"topology"` / `"trajectory"`). |
| `file_format` | File extension without the dot. |
| `request` | The request string this plan was made for. |
| `n_atoms` | Estimated atoms matching the domain's query. |
| `n_frames` | Frame count for this read (always 1 for static files). |
| `bytes_per_atom_per_frame` | Byte cost per atom per frame for this request. |
| `estimated_bytes` | `n_atoms x n_frames x bytes_per_atom_per_frame`. |
| `confidence` | (stochastic only) `"none"`, `"low"`, `"medium"`, `"high"`. |
| `sampling` | (stochastic only) sampling diagnostics. |


## 5. The `combined` estimate

When you query multiple domains in one `fetch()` call, each per-domain plan
is computed in isolation. But the actual returned payload reflects the
**intersection** of selections across domains — typically far smaller than
any individual plan suggests.

`plan["combined"]` resolves this by reporting:

- `n_atoms_upper_bound` — the minimum `n_atoms` across all per-domain plans.
  The true intersection cannot exceed this.
- `total_estimated_bytes` — sum across requested domains, using the upper
  bound and each domain's frame count and per-atom cost.

This is your honest top-bound memory estimate.


In [7]:
out = s.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TYPE_R='residue_names',
    TOPO_R='atom_types',
    TRAJ_R='positions',
    devFlag=True,
)

print('per-domain n_atoms:')
for d in ('typing', 'topology', 'trajectory'):
    if d in out['plan']:
        print(f'  {d:11s} {out["plan"][d]["n_atoms"]}')

print('combined         :', out['plan']['combined'])


per-domain n_atoms:
  typing      3
  topology    24
  trajectory  20
combined         : {'n_atoms_upper_bound': 3, 'total_estimated_bytes': 252}


The smallest per-domain `n_atoms` (here, `typing` with 3 CA atoms) becomes
the upper bound. `total_estimated_bytes` then sums each requested domain's
contribution at that bound — typing's `residue_names`, topology's
`atom_types`, trajectory's `positions x 5 frames` — giving you a single
worst-case figure.


## 6. `planFlag` — get the plan, skip the read

When you only want the cost estimate without paying for the actual read,
combine `devFlag=True` with `planFlag=True`. The envelope comes back with
`payload=None` and the planner has done all of its work.


In [8]:
out = s.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TYPE_R='residue_names',
    TRAJ_R='positions',
    devFlag=True,
    planFlag=True,
)

print('payload:', out['payload'])
print('combined:', out['plan']['combined'])


payload: None
combined: {'n_atoms_upper_bound': 3, 'total_estimated_bytes': 204}


**Note:** `planFlag` only takes effect when `devFlag=True`. Without `devFlag`
it's silently ignored — the basic API stays unconditional and simple. This is
deliberate: planning is a developer / advanced feature, gated behind `devFlag`.


## 7. Discovering what's available

Three programmatic accessors give you the same information `print_info()`
shows.


In [9]:
print('Typing keywords:', sorted(s.get_type_keys()))
print('Typing requests:', sorted(s.get_type_reqs()))
print()
print('Topology keywords:', sorted(s.get_topo_keys()))
print('Topology requests:', sorted(s.get_topo_reqs()))
print()
print('Trajectory keywords:', sorted(s.get_traj_keys()))
print('Trajectory requests:', sorted(s.get_traj_reqs()))


Typing keywords: ['atom_name', 'global_ids', 'local_ids', 'occupancy', 'residue_ids', 'residue_name', 'segment_name', 'temperature_coeff', 'x', 'y', 'z']
Typing requests: ['atom_names', 'global_ids', 'local_ids', 'occupancy', 'positions', 'property-box_size', 'property-number_of_atoms', 'property-number_of_residues', 'property-number_of_segments', 'residue_ids', 'residue_names', 'segment_names', 'temperature_coeff', 'x', 'y', 'z']

Topology keywords: ['atom_name', 'atom_type', 'bonded_with', 'bonded_with_mode', 'charge', 'global_ids', 'is_virtual', 'local_ids', 'mass', 'residue_ids', 'residue_name', 'segment_name']
Topology requests: ['atom_names', 'atom_types', 'bonds_with', 'charges', 'global_ids', 'local_ids', 'masses', 'property-system_charge', 'residue_ids', 'residue_names', 'segment_names']

Trajectory keywords: ['frame_interval', 'global_ids']
Trajectory requests: ['global_ids', 'positions']


**When to use each:**

- `print_info()` — interactive exploration, one-shot human-readable overview.
- `get_*_keys()` / `get_*_reqs()` — programmatic checks (e.g. "does this file
  support `velocities`?" before issuing the call).


## 8. Memory budgeting workflow

A practical use of the planner: decide whether a read is feasible before
committing to it.


In [10]:
BUDGET_BYTES = 10_000_000   # 10 MB

# Plan first (no payload read).
plan_only = s.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TRAJ_R='positions',
    devFlag=True,
    planFlag=True,
)

estimated = plan_only['plan']['combined']['total_estimated_bytes']
print(f'Estimated cost: {estimated:,} bytes')

if estimated <= BUDGET_BYTES:
    print('Within budget - executing.')
    result = s.fetch(
        TYPE_Q={'atom_name': 'CA'},
        TRAJ_R='positions',
    )
    print('payload shape:', result['trajectory'].shape)
else:
    print('Over budget - reduce the query or stream in chunks.')


Estimated cost: 1,200 bytes
Within budget - executing.
payload shape: (5, 3, 3)


## 9. Graph-pattern matching with recursive `bonded_with`

A `bonded_with` neighbour sub-query can itself contain `bonded_with`. This
enables graph-pattern matching — selecting atoms by patterns that extend
more than one bond into the graph.

Trivial case: "find atoms that are **hydrogens bonded to a carbon**."

```python
TOPO_Q={
    'atom_type': 'H',
    'bonded_with': {'neighbor': {'atom_type': 'C'}, 'count': {'ge': 1}},
}
```

Two-hop case: "find atoms **bonded to an α-carbon that is itself bonded to
a backbone nitrogen**." The outer query asks for one-hop neighbours of
atoms that pass an inner predicate — and the inner predicate is itself a
bond-graph pattern.

```python
TOPO_Q={
    'bonded_with': {
        'neighbor': {
            'atom_type': 'CT1',
            'bonded_with': {'neighbor': {'atom_type': 'NH1'}, 'count': {'ge': 1}},
        },
        'count': {'ge': 1},
    },
}
```

You can nest to any realistic depth. The parser caches neighbour
resolutions within a single user call, so identical sub-patterns at any
depth are resolved exactly once. A safety ceiling of 16 levels prevents
runaway depth from malformed queries; exceeding it raises `RecursionError`.


In [11]:
# Real example on the synthetic system:
# "Atoms bonded to an OT that is itself bonded to at least 2 HT atoms."
# The inner pattern picks out water oxygens (each has 2 HT bonds);
# the outer picks their neighbours. Result: the 4 water hydrogens.

result = s.fetch(
    TOPO_Q={
        'bonded_with': {
            'neighbor': {
                'atom_type': 'OT',
                'bonded_with': {
                    'neighbor': {'atom_type': 'HT'},
                    'count': {'ge': 2},
                },
            },
            'count': {'ge': 1},
        },
    },
    TOPO_R='atom_names',
)
print('Atoms bonded to a valid water O:', result['topology'])


Atoms bonded to a valid water O: ['H1', 'H2', 'H1', 'H2']


## 10. `updateFlag` and global system properties

`sim` carries a small dict of system-level facts populated when files load.
You can read it directly, or use `add_info()` to record facts the parser
doesn't surface (e.g. ensemble type, simulation name).


In [12]:
print('Auto-populated from files:')
pprint.pp({k: v for k, v in s.global_system_properties.items() if v is not None})

# Add facts the parser doesn't know about.
s.add_info({
    'sim_name':        'my_test_run',
    'ensemble_type':   'NPT',
    'timestep':        2.0,   # fs
})

print()
print('After add_info:')
pprint.pp({k: v for k, v in s.global_system_properties.items() if v is not None})


Auto-populated from files:
{'num_atoms': 20,
 'num_residues': 5,
 'num_frames': 5,
 'start_box_size': (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)}

After add_info:
{'sim_name': 'my_test_run',
 'timestep': 2.0,
 'num_atoms': 20,
 'num_residues': 5,
 'num_frames': 5,
 'ensemble_type': 'NPT',
 'start_box_size': (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)}


`updateFlag=True` (passable to `positions()`, `fetch()`, `select()`,
`get_types()`, etc.) re-runs the parser's `_update_*_globals_*` function
and re-syncs the global properties. Useful if you suspect a file may have
changed underneath you, or after adding a new file mid-session.


## What next?

You've now seen everything the user-facing API exposes. If you want to
understand how the parsers and planner are wired together, or want to add
support for a new file format, see **`tutorial_dev.ipynb`**.
